# Milestone 1: Data Profiling

Profil awal dataset besar Indonesia menggunakan Polars LazyFrame dan DuckDB. Isi sumber serta deskripsi dataset pada `data/README.md` sebelum menjalankan notebook ini.

In [2]:
from pathlib import Path

import duckdb
import polars as pl

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_PATH = PROJECT_ROOT / 'data' / 'raw' / 'dataset.parquet'

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f'Dataset tidak ditemukan: {DATA_PATH}. '
        'Unduh data ke data/raw/ lalu sesuaikan DATA_PATH.'
    )

print(f'Dataset: {DATA_PATH.name}')
print(f'Ukuran file: {DATA_PATH.stat().st_size / 1_000_000:.1f} MB')

Dataset: dataset.parquet
Ukuran file: 846.4 MB


## Profil Polars

`scan_parquet` hanya membangun rencana query; file baru dibaca ketika `collect()` dipanggil.

In [3]:
lazy_data = pl.scan_parquet(DATA_PATH)
print(lazy_data.collect_schema())

ringkasan = lazy_data.select([
    pl.len().alias('jumlah_baris'),
    pl.all().null_count().name.suffix('_null'),
]).collect()
ringkasan

Schema({'osm_type': String, 'osm_id': Int64, 'name': String, 'category': String, 'lat': Float64, 'lon': Float64, 'tags_json': Binary})


jumlah_baris,osm_type_null,osm_id_null,name_null,category_null,lat_null,lon_null,tags_json_null
u32,u32,u32,u32,u32,u32,u32,u32
51842020,0,0,50582209,6537382,0,0,0


In [4]:
contoh_data = lazy_data.head(10).collect()
contoh_data

osm_type,osm_id,name,category,lat,lon,tags_json
str,i64,str,str,f64,f64,binary
"""way""",809958596,"""Bayu-Undan to Darwin Gas Pipel…","""man_made=pipeline""",-12.262813,130.390424,"b""{""fixme"":""position\x20approx\x20and\x20northern\x20end\x20not\x20exact"",""layer""…"
"""node""",2400320280,null,null,-11.01,122.876944,"b""{""source"":""http://www.un.org/Depts/los/LEGISLATIONANDTREATIE""…"
"""node""",10912953441,"""Southernmost point of Asia""","""tourism=attraction""",-11.009191,122.876322,"b""{""alt_name"":""Southernmost\x20point\x20of\x20Indonesia"",""alt_name:fr"":""…"
"""way""",706574359,null,null,-10.998834,122.878625,"b""{""highway"":""track"",""source"":""digitalglobe"",""unpaved"":""yes""}"""
"""way""",682004460,null,"""natural=water""",-10.99312,122.877212,"b""{""natural"":""water""}"""
"""way""",682004462,null,"""natural=water""",-10.988591,122.866896,"b""{""natural"":""water""}"""
"""way""",675293216,null,"""natural=beach""",-10.987799,122.870621,"b""{""natural"":""beach""}"""
"""way""",682002770,null,"""building=yes""",-10.985967,122.851335,"b""{""building"":""yes""}"""
"""way""",682002773,null,"""building=yes""",-10.985746,122.85142,"b""{""building"":""yes""}"""


## Profil DuckDB

DuckDB membaca file Parquet langsung dari disk dan menghasilkan ringkasan statistik setiap kolom.

In [5]:
connection = duckdb.connect()
try:
    profil_duckdb = connection.sql(
        'SUMMARIZE SELECT * FROM read_parquet(?)', params=[str(DATA_PATH)]
    ).pl()
except AttributeError:
    profil_duckdb = connection.execute(
        'SUMMARIZE SELECT * FROM read_parquet(?)', [str(DATA_PATH)]
    ).pl()
profil_duckdb

column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
str,str,str,str,i64,str,str,str,str,str,i64,"decimal[9,2]"
"""osm_type""","""VARCHAR""","""node""","""way""",3,null,null,null,null,null,51842020,0.00
"""osm_id""","""BIGINT""","""30865""","""14304530641""",52608291,"""880079071.7982199""","""1171351931.35747""","""553158699""","""646386740""","""958821254""",51842020,0.00
"""name""","""VARCHAR""","""""At06"" Complex""","""🐠🐟🐡🏡""",602978,null,null,null,null,null,51842020,97.57
"""category""","""VARCHAR""","""aeroway=aerodrome""","""tourism=zoo""",2405,null,null,null,null,null,51842020,12.61
"""lat""","""DOUBLE""","""-12.26281288888889""","""12.890181635483872""",50163561,"""-5.203345193393486""","""3.6012937764495225""","""-7.540316674165635""","""-6.789303823232048""","""-3.4976930085796587""",51842020,0.00
"""lon""","""DOUBLE""","""89.14424023333333""","""142.499998""",51865839,"""110.94927122850403""","""7.103823093384553""","""106.93728482745384""","""110.13618773557577""","""114.54633321281419""",51842020,0.00
"""tags_json""","""JSON""","""{""2.5"":""yes"",""highway"":""living…","""{""xmas:feature"":""tree""}""",1141708,null,null,null,null,null,51842020,0.00


## Catatan Milestone 1

### 1. Ringkasan Skala dan Metadata Dataset
- **Nama Dataset**: Indonesia POI: OpenStreetMap Detail (ekstraksi Geofabrik OpenStreetMap Indonesia via Hugging Face `Lyon28/indonesia-poi-openStreetMaps`).
- **Format Penyimpanan**: Apache Parquet (`dataset.parquet`).
- **Ukuran File**: **846,4 MB** (terkompresi columnar Parquet; setara >3,5 GB dalam format CSV mentah).
- **Jumlah Baris**: **51.842.020 baris** (51,8 juta baris), jauh melampaui ketentuan batas minimal 1.000.000 baris.
- **Jumlah Atribut**: 7 kolom (`osm_type`, `osm_id`, `name`, `category`, `lat`, `lon`, `tags_json`).

### 2. Skema Kolom dan Tipe Data
- `osm_type` (String / VARCHAR): Tipe geometri OpenStreetMap (`node`, `way`, `relation`). Tidak memiliki missing values (0 null).
- `osm_id` (Int64 / BIGINT): Identifier unik numerik entitas OpenStreetMap. Tidak memiliki missing values (0 null).
- `name` (String / VARCHAR): Nama spesifik lokasi/fasilitas publik (misal nama gedung, sekolah, toko, restoran).
- `category` (String / VARCHAR): Kategori klasifikasi fasilitas (misal: `amenity=school`, `amenity=place_of_worship`, `shop=convenience`, `tourism=attraction`). Memiliki 2.405 variasi kategori unik.
- `lat`, `lon` (Float64 / DOUBLE): Koordinat spasial lintang dan bujur bereferensi WGS84. Tidak memiliki missing values (0 null).
- `tags_json` (String JSON): Kumpulan metadata tag OpenStreetMap dalam bentuk key-value JSON ter-serialize. Tidak memiliki missing values (0 null).

### 3. Analisis Missing Values (Nilai Null)
- **Atribut Lengkap (0% Null)**: `osm_type`, `osm_id`, `lat`, `lon`, dan `tags_json` terisi 100% pada seluruh 51.842.020 baris.
- **`category` (12,61% Null)**: Sebanyak 6.537.382 baris tidak memiliki kategori primer di kolom `category`. Objek-objek ini sebagian besar merupakan fitur fisik umum (seperti batas wilayah, kontur alam, atau infrastruktur utilitas) yang tag spesifiknya tersimpan di dalam `tags_json`.
- **`name` (97,57% Null)**: Sebanyak 50.582.209 baris tidak memiliki nama spesifik, dan hanya 1.259.811 baris (~2,43%) yang memiliki nama entitas. Hal ini merupakan karakteristik wajar dari data OpenStreetMap di mana mayoritas segmen jalan, batas lahan, atau infrastruktur fisik tidak diberi nama tersendiri.

### 4. Analisis Statistik Geospasial dan Deteksi Anomali Awal
- **Distribusi Koordinat**:
  - Lintang (`lat`): Minimum -12,2628°, Maksimum 12,8902°, Rata-rata -5,2033°, Median (q50) -6,7987°.
  - Bujur (`lon`): Minimum 89,1442°, Maksimum 142,5000°, Rata-rata 110,9493°, Median (q50) 110,1349°.
- **Deteksi Outlier Spasial**:
  - Batas kedaulatan administratif kepulauan Indonesia berada pada rentang latitude kira-kira -11,0° (Pulau Rote) s.d. 6,1° (Pulau Weh/Sabang), dan longitude 95,0° s.d. 141,0° (Merauke).
  - Nilai ekstrim seperti `lat` hingga 12,89° dan `lon` hingga 89,14° / 142,50° mengindikasikan adanya objek penyangga (*buffer boundary*) dari ekstrak Geofabrik, mencakup perairan internasional, pulau terluar, serta pipa/kabel bawah laut lintas teritorial (misal: *Bayu-Undan to Darwin Gas Pipeline* pada koordinat lat -12,26°).

### 5. Rencana Aksi Pembersihan dan Validasi Data untuk Milestone 2
1. **Filtering Bounding Box Spasial Indonesia**:
   Menerapkan filter rentang geografis (-11,5 ≤ `lat` ≤ 6,5 dan 95,0 ≤ `lon` ≤ 141,5) untuk memastikan entitas yang dianalisis benar-benar berada di dalam teritorial Indonesia.
2. **Strategi Penanganan Missing Values**:
   - Untuk analisis fasilitas publik (POI), lakukan filter atau sub-sampling pada data yang memiliki `category` terisi atau lakukan parsing fallback terhadap tag tertentu di `tags_json`.
   - Mengisi label eksplisit `[Tanpa Nama]` atau kategori `unclassified` untuk entitas yang tidak memiliki nama agar tidak mengganggu agregasi visualisasi.
3. **Normalisasi & Ekstraksi Tag JSON**:
   Mengekstrak atribut administratif (seperti nama pulau, kota, provinsi, tipe jalan, atau kapasitas) dari kolom `tags_json` menggunakan ekspresi JSON Polars yang efisien.
4. **Deduplikasi ID**:
   Melakukan validasi duplikasi `osm_id` pada tipe geometri yang sama untuk menjamin integritas unit analisis.